# Training loop: learn $y = 2x + 1$

Template we reuse for almost every project:

- synthetic data → `TensorDataset` / `DataLoader`
- `model.train()` vs `model.eval()`
- zero grad → forward → loss → backward → step
- validation under `torch.no_grad()`
- save **weights** (`state_dict`) vs a **checkpoint** (weights + optimizer + epoch)


## 1. Imports and hyperparameters


In [ ]:
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(0)

learning_rate = 1e-2
num_epochs = 100
batch_size = 16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

artifacts = Path("artifacts")
artifacts.mkdir(exist_ok=True)


## 2. Synthetic data


In [ ]:
true_weight = torch.tensor([[2.0]])
true_bias = torch.tensor([1.0])

x_train = torch.randn(100, 1) * 5
y_train = true_weight * x_train + true_bias + torch.randn(100, 1) * 0.5

x_val = torch.randn(20, 1) * 5
y_val = true_weight * x_val + true_bias + torch.randn(20, 1) * 0.5

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=batch_size, shuffle=False)


## 3. Model, loss, optimizer


In [ ]:
model = nn.Linear(1, 1).to(device)
loss_fn = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate)


## 4. Training


In [ ]:
print("Starting training...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    num_batches = 0

    for features, labels in train_loader:
        features = features.to(device)
        labels = labels.to(device)

        outputs = model(features)
        loss = loss_fn(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        num_batches += 1

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch + 1}/{num_epochs}]  train loss: {running_loss / num_batches:.4f}")


## 5. Evaluation


In [ ]:
model.eval()
total_val_loss = 0.0
num_val_batches = 0

with torch.no_grad():
    for features, labels in val_loader:
        features, labels = features.to(device), labels.to(device)
        loss = loss_fn(model(features), labels)
        total_val_loss += loss.item()
        num_val_batches += 1

print(f"Validation loss: {total_val_loss / num_val_batches:.4f}")
for name, param in model.named_parameters():
    print(f"{name}: {param.data.squeeze()}")
print(f"(true weight: {true_weight.item():.4f}, true bias: {true_bias.item():.4f})")


## 6. Save and load weights

`state_dict()` is just the learned tensors. We must reconstruct the **same** architecture before `load_state_dict`.


In [ ]:
weights_path = artifacts / "linear_regression_model.pt"
torch.save(model.state_dict(), weights_path)

loaded = nn.Linear(1, 1).to(device)
loaded.load_state_dict(torch.load(weights_path, map_location=device, weights_only=True))
loaded.eval()

with torch.no_grad():
    sample = torch.tensor([[10.0]], device=device)
    print(f"prediction for 10.0: {loaded(sample).item():.4f}  (target ~ 21)")


## 7. Pause training and continue later

Two different things get confused here.

**A. Weights only (`state_dict`)** — good for inference, or to start a *new* training run from those weights. We rebuild a fresh optimizer. Momentum / Adam moments are **lost**.

**B. Full checkpoint** — what we want to *pause* and resume later. Save the model, the optimizer, and the epoch. Then we continue from the exact same step.


In [ ]:
# --- A. Weights only: continue, but optimizer history is gone ---
fresh_opt = optim.SGD(loaded.parameters(), lr=learning_rate)
print("Method A: model weights reloaded, optimizer is brand new.")

# --- B. Full checkpoint: pause here, resume later ---
checkpoint_path = artifacts / "checkpoint.pt"
checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "epoch": num_epochs,
}
torch.save(checkpoint, checkpoint_path)
print(f"Paused at epoch {num_epochs}. Saved {checkpoint_path}")


Later (another day, another process): rebuild the **same** architecture, reload the checkpoint, then keep training from `epoch + 1`.


In [ ]:
resumed = nn.Linear(1, 1).to(device)
resumed_opt = optim.SGD(resumed.parameters(), lr=learning_rate)

ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
resumed.load_state_dict(ckpt["model_state_dict"])
resumed_opt.load_state_dict(ckpt["optimizer_state_dict"])
start_epoch = ckpt["epoch"] + 1
extra_epochs = 20
total_epochs = ckpt["epoch"] + extra_epochs

print(f"Resuming from epoch {start_epoch} up to {total_epochs}...")

for epoch in range(start_epoch, total_epochs + 1):
    resumed.train()
    running_loss, num_batches = 0.0, 0
    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)
        resumed_opt.zero_grad()
        loss = loss_fn(resumed(features), labels)
        loss.backward()
        resumed_opt.step()
        running_loss += loss.item()
        num_batches += 1
    if epoch % 10 == 0 or epoch == total_epochs:
        print(f"Epoch [{epoch}/{total_epochs}]  train loss: {running_loss / num_batches:.4f}")

resumed.eval()
with torch.no_grad():
    print(f"weight after resume: {resumed.weight.data.squeeze()}")
    print(f"bias after resume:   {resumed.bias.data.squeeze()}")
